## mindspore.Tensor.max() -> Tensor
返回输入Tensor的最大值。
## mindspore.Tensor.max(dim, keepdim=False) -> tuple(Tensor)
按指定维度返回输入Tensor的最大值及其索引。
## mindspore.Tensor.max(axis=None, keepdims=False, *, initial=None, where=True, return_indices=False) -> Tensor or scalar
返回输入Tensor的最大值或最大值及索引。

1、参数比较：  
| mindspore   |              |              |
| :----:      | :----:      | :----:      |
| 重载1       | 重载2       | 重载3         |
|             | dim         | axis        |
|             | keepdim     | keepdims    |
|             |             | initial     |
|             |             | where       |
|             |             | return_indices|


| torch       |              |            |
| :----:      | :----:      | :----:      |
| 重载1       | 重载2       | 重载3         |
|             | dim         | other       |
|             | keepdim     | out         |
|             | out         |             |

| jax       |
| :----:      |
| axis        |
| out         |
| keepdims    |
| initial     |
| where       |


2、返回值比较

重载1：

In [5]:
import numpy as np
import mindspore as ms
import torch
import jax

input = np.array([[9, 3, 4, 5],
                  [5, 2, 7, 4],
                  [8, 1, 3, 6]])

y1 = ms.tensor(input)
y1_output = y1.max()
y2 = torch.tensor(input)
y2_output = y2.max()
y3_output = input.max()
print ('mindspore output:\n',y1_output)
print('\n')
print ('torch output:\n',y2_output)
print('\n')
print ('jax output:\n',y3_output)

mindspore output:
 9


torch output:
 tensor(9)


jax output:
 9


* ms与jax不返回类型。

重载2：

In [4]:
y1 = ms.tensor(input)
y1_output = y1.max(dim=1, keepdim=True)
y2 = torch.tensor(input)
y2_output = y2.max(dim=1, keepdim=True)
y3 = input
y3_output = y3.max(axis=1, keepdims=True)
print ('mindspore output:\n',y1_output)
print('\n')
print ('torch output:\n',y2_output)
print('\n')
print ('jax output:\n',y3_output)

mindspore output:
 (Tensor(shape=[3, 1], dtype=Int64, value=
[[9],
 [7],
 [8]]), Tensor(shape=[3, 1], dtype=Int64, value=
[[0],
 [2],
 [0]]))


torch output:
 torch.return_types.max(
values=tensor([[9],
        [7],
        [8]]),
indices=tensor([[0],
        [2],
        [0]]))


jax output:
 [[9]
 [7]
 [8]]


* jax不返回最大值的索引。

重载3：ms与jax

In [9]:
where=np.array([[0, 0, 0, 0],
                [0, 1, 0, 1],
                [1, 1, 1, 0]], dtype=bool)

y1 = ms.tensor(input)
y1_output = y1.max(axis=1, keepdims=True, initial=0, where=ms.tensor(where), return_indices=True)
y3 = input
y3_output = y3.max(axis=1, keepdims=True, initial=0, where=ms.tensor(where))
print ('mindspore output:\n',y1_output)
print('\n')
print ('jax output:\n',y3_output)

mindspore output:
 (Tensor(shape=[3, 1], dtype=Int64, value=
[[0],
 [4],
 [8]]), Tensor(shape=[3, 1], dtype=Int64, value=
[[0],
 [3],
 [0]]))


jax output:
 [[0]
 [4]
 [8]]


* jax不返回最大值的索引。

In [19]:
other=np.array([[10, 0, 0, 0],
                [0, 1, 0, 1],
                [1, 1, 5, 0]])
y2 = torch.tensor(input)
y2_output = y2.max(torch.tensor(other))
print ('torch output:\n',y2_output)

torch output:
 tensor([[10,  3,  4,  5],
        [ 5,  2,  7,  4],
        [ 8,  1,  5,  6]])


3、报错信息比较

In [16]:
y1_output = y1.max(dim=(1,2), keepdim=True)

TypeError: Failed calling max with "max(dim=Tuple<int, int>, keepdim=bool)".
The valid calling should be:
"Tensor.max()"
"Tensor.max(dim=<int>, keepdim=<bool>)"
"Tensor.max(axis=<int,Tuple,None,List>, keepdims=<bool>, *, initial=<number,None>, where=<bool,Tensor>, return_indices=<bool>)"


----------------------------------------------------
- C++ Call Stack: (For framework developers)
----------------------------------------------------
mindspore/ccsrc/pipeline/pynative/op_function/converter.h:112 Parse


In [17]:
y2_output = y2.max(dim=(1,2), keepdim=True)

TypeError: max() received an invalid combination of arguments - got (keepdim=bool, dim=tuple, ), but expected one of:
 * ()
 * (Tensor other)
 * (int dim, bool keepdim)
      didn't match because some of the arguments have invalid types: (!dim=tuple of (int, int)!, keepdim=bool, )
 * (name dim, bool keepdim)
      didn't match because some of the arguments have invalid types: (!dim=tuple of (int, int)!, keepdim=bool, )


In [6]:
y3_output = y1.max(dim=(1,2), keepdim=True)

TypeError: tile requires ndarray or scalar arguments, got <class 'list'> at position 0.

当输入类型不正确时，报错信息torch简洁明确。建议ms优化。